# Spaceship Titanic — Предсказание переноса пассажиров

**Тема:** Бинарная классификация с категориальными признаками  
**Инструменты:** `pandas`, `CatBoostClassifier`

---

## О задаче

Космический лайнер *Spaceship Titanic* столкнулся с аномалией, и часть пассажиров была перенесена в другое измерение. Нужно предсказать, кто из пассажиров был перенесён (`Transported` — True/False).

**Почему CatBoost:** в данных много категориальных признаков (планета, пункт назначения, крио-сон). CatBoost умеет работать с категориями напрямую — не нужно вручную кодировать их в числа.

## 1. Загрузка данных

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

train = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test  = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

print('Размер train:', train.shape)
print('Размер test: ', test.shape)
train.head()

## 2. Признаки и целевая переменная

Убираем колонки, которые не помогают модели: `Name`, `PassengerId`, `Cabin` (сырой текст). Целевая переменная — `Transported`, переводим её в 0/1.

Категориальные признаки: `HomePlanet`, `CryoSleep`, `Destination`, `VIP`.

In [ ]:
cat_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
drop_cols = ['Transported', 'Name', 'PassengerId', 'Cabin']

X = train.drop(columns=drop_cols)
y = train['Transported'].astype(int)
X_test = test.drop(columns=['Name', 'PassengerId', 'Cabin'])

## 3. Обработка пропусков

В числовых колонках пропуски заполняем «маркером» `-999` (CatBoost воспримет это как отдельный сигнал). В категориальных — приводим к строке и заполняем `'None'`.

In [ ]:
for df in [X, X_test]:
    numeric_features = df.select_dtypes(include=['float64', 'int64']).columns
    df[numeric_features] = df[numeric_features].fillna(-999)
    df[cat_features] = df[cat_features].astype(str).fillna('None')

print('Пропусков не осталось:', int(X.isnull().sum().sum()) == 0)

## 4. Обучение модели

Делим train на обучающую и валидационную части (80/20), чтобы честно оценить качество. `early_stopping_rounds=50` останавливает обучение, если качество на валидации перестало расти — защита от переобучения.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=6,
    eval_metric='Accuracy',
    random_seed=42,
    early_stopping_rounds=50,
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    verbose=100,
)

## 5. Оценка качества

Accuracy на валидации + важность признаков (какие признаки модель считает главными).

In [ ]:
val_acc = accuracy_score(y_val, model.predict(X_val))
print(f'Accuracy на валидации: {val_acc:.4f}')
print()

importances = sorted(zip(X.columns, model.feature_importances_), key=lambda x: x[1], reverse=True)
print('Важность признаков:')
for name, imp in importances:
    print(f'  {name:<16}: {imp:.2f}')

## 6. Генерация сабмита

In [ ]:
predictions = model.predict(X_test).astype(bool)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': predictions,
})
submission.to_csv('submission.csv', index=False)
print('submission.csv сохранён!')
submission.head()

## Итог

| Шаг | Что сделали |
|---|---|
| Данные | Загрузили, убрали текстовые/ID-колонки |
| Пропуски | Числовые → `-999`, категориальные → `'None'` |
| Модель | CatBoost с категориальными признаками напрямую |
| Валидация | 80/20 split + early stopping |
| Метрика | Accuracy |

**Ключевая идея:** CatBoost избавляет от ручного кодирования категорий и сам находит хорошее разбиение — быстрый и сильный бейзлайн для табличных данных со смешанными типами.